#  Tầng 2a: Data Cleanup

**Mục tiêu:**
1. Đổi tên cột câu hỏi dài → mã ngắn gọn (Q1, Q2... + tên biến tiếng Anh)
2. Xử lý missing values
3. Chuẩn hóa kiểu dữ liệu

**Input:** `interim/responses.csv`  
**Output:** `processed/cleaned_responses.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from pathlib import Path
# Ensure paths resolve from the project root when notebooks run from notebooks/.
import os
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
# ── Config ──────────────────────────────────────────────
INPUT_FILE = "interim/responses.csv"
OUTPUT_FILE = "processed/cleaned_responses.csv"

assert Path(INPUT_FILE).exists(), f" Chưa có {INPUT_FILE}. Hãy chạy 0_data_ingestion.ipynb trước!"
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
print(f" Đọc {INPUT_FILE}: {df.shape[0]} dòng × {df.shape[1]} cột")

In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 1: RENAME - Mapping cột gốc → mã ngắn gọn
# ══════════════════════════════════════════════════════════

COLUMN_MAPPING = {
    # ── Thông tin cá nhân ──
    "Dấu thời gian":                                                         "timestamp",
    "Bạn đang học tại trường nào?":                                           "Q1_university",
    "Bạn đang học năm mấy?":                                                  "Q2_year",
    "Chuyên ngành hướng đào tạo chính của bạn?":                              "Q3_major",
    "Lĩnh vực / Sản phẩm chính mà bạn đang thực hiện?":                      "Q4_product_field",

    # ── Nhận thức CI/CD ──
    "Bạn đã từng nghe đến thuật ngữ CI/CD(Continuous Integration / Continuous Deployment) chưa?":  "Q5_cicd_awareness",
    "Mức độ tự động hóa trong các dự án lập trình mà bạn từng thực hiện là gì?  ":                "Q6_automation_level",

    # ── Sử dụng công cụ (Checkbox - nhiều đáp án) ──
    "Bạn đã từng sử dụng công cụ CI/CD nào trong quá trình học tập hoặc làm dự án cá nhân? ":     "Q7_tools_used",
    "Bạn đã tiếp cận CI/CD chủ yếu thông qua nguồn nào?":                                        "Q8_learning_source",
    "Bạn sử dụng các công cụ CI/CD đã chọn cho những mục đích nào trong quy trình phát triển phần mềm?": "Q9_usage_purpose",
    "Theo bạn, CI/CD giúp cải thiện điều gì trong quy trình phát triển phần mềm?":                "Q10_cicd_benefits",

    # ── DORA Metrics ──
    "Trong các dự án lập trình mà bạn từng thực hiện (đồ án, project cá nhân, hoặc project nhóm), bạn cập nhật phiên bản mới của ứng dụng lên môi trường chạy thử (server, hosting, cloud, v.v.) với tần suất như thế nào? ": "Q11_deploy_frequency",
    "Kể từ thời điểm bạn hoàn thành việc viết code và đẩy lên hệ thống (commit), mất bao lâu để những thay đổi đó thực sự chạy thành công trên môi trường cho người dùng cuối (Production)?  ": "Q12_lead_time",
    "Khi dự án của bạn gặp lỗi nghiêm trọng sau khi cập nhật code (ví dụ: ứng dụng không chạy, lỗi server, lỗi chức năng), bạn thường mất bao lâu để khắc phục và đưa hệ thống hoạt động bình thường trở lại? ": "Q13_recovery_time",
    "Trong các lần bạn cập nhật hoặc triển khai phiên bản mới của dự án (đồ án, project cá nhân, hoặc project nhóm), tỷ lệ gặp lỗi nghiêm trọng khiến hệ thống không hoạt động đúng thường ở mức nào? ": "Q14_failure_rate",

    # ── Likert Scale (1-5) ──
    "CI/CD giúp tôi tiết kiệm thời gian triển khai trong các dự án học tập":                      "Q15_save_time",
    "CI/CD giúp tôi phát hiện lỗi code sớm hơn so với kiểm tra thủ công  ":                       "Q16_early_bug_detect",
    "Biết sử dụng CI/CD giúp tôi tự tin hơn khi thực tập hoặc xin việc  ":                        "Q17_job_confidence",
    "Tôi thấy việc tìm hiểu CI/CD không quá khó so với các kỹ năng lập trình khác  ":             "Q18_ease_of_learning",
    "Tôi có thể tự học CI/CD thông qua tài liệu và hướng dẫn có sẵn trên mạng  ":                 "Q19_self_study",
    "Tôi có thể tự thiết lập một pipeline CI/CD đơn giản sau khi đọc hướng dẫn  ":                 "Q20_setup_ability",
    " Bạn bè / đồng học trong nhóm dự án khuyến khích tôi sử dụng CI/CD  ":                       "Q21_peer_influence",
    "Giảng viên hoặc mentor trong quá trình thực tập đề cao việc áp dụng CI/CD  ":                 "Q22_mentor_influence",
    "CI/CD được coi là kỹ năng tiêu chuẩn mà một lập trình viên chuyên nghiệp cần có ":           "Q23_industry_standard",
    "Chương trình đào tạo tại trường cung cấp đủ kiến thức nền để tôi tiếp cận CI/CD":            "Q24_curriculum_readiness",
    "Các công cụ CI/CD miễn phí (GitHub Actions, GitLab CI...) đủ để tôi thực hành trong môi trường học tập  ": "Q25_free_tools_sufficient",
    "Tôi dễ dàng tìm được sự hỗ trợ khi gặp khó khăn với CI/CD (giảng viên, cộng đồng online, tài liệu...)  ": "Q26_support_access",
    "Tôi có ý định áp dụng CI/CD vào các dự án học tập hoặc thực tập sắp tới  ":                  "Q27_intent_to_adopt",
    "Tôi dự định tự học thêm về CI/CD ngoài chương trình đào tạo chính khoá  ":                   "Q28_self_learn_plan",
    "Tôi sẽ ưu tiên tham gia các dự án hoặc nhóm có sử dụng CI/CD  ":                             "Q29_prefer_cicd_projects",
    "Tôi đã thực tế cấu hình hoặc sử dụng CI/CD trong ít nhất một dự án  ":                       "Q30_actual_usage",
    "Tôi thường xuyên dùng CI/CD trong quá trình phát triển phần mềm của mình  ":                  "Q31_regular_usage",
    "Tôi chủ động đề xuất hoặc thiết lập CI/CD cho nhóm mà không cần ai yêu cầu  ":               "Q32_proactive_setup",

    # ── Checkbox & Free-text cuối survey ──
    "Trong quá trình học tập hoặc thực hiện các dự án lập trình (đồ án, project cá nhân, project nhóm), bạn cảm thấy khó khăn ở những giai đoạn nào của quy trình CI/CD? ": "Q33_cicd_difficulties",
    "Theo bạn, những khó khăn hoặc thách thức nào khiến sinh viên khó áp dụng CI/CD trong quá trình học tập và làm dự án? ": "Q34_adoption_barriers",

    # ── Open-ended (free text) ──
    "Theo bạn, điều gì cần được cải thiện để sinh viên dễ tiếp cận và học CI/CD hiệu quả hơn?  ": "Q35_improvement_suggestions",
    "Bạn có đề xuất nào để các trường đại học hoặc giảng viên hỗ trợ sinh viên học và áp dụng CI/CD tốt hơn trong các môn học hoặc dự án?  ": "Q36_university_suggestions",
    "Theo bạn, sinh viên cần được trang bị thêm những kiến thức hoặc kỹ năng nào để áp dụng CI/CD hiệu quả trong các dự án phần mềm?  ": "Q37_skills_needed",

    # ── Single-choice cuối ──
    "Bạn có dự định tìm hiểu hoặc áp dụng CI/CD trong các dự án lập trình (đồ án, project cá nhân, project nhóm) trong thời gian tới không? ": "Q38_future_plan",
    "Theo bạn, đâu là rào cản lớn nhất khiến sinh viên chưa áp dụng CI/CD trong các dự án lập trình? ": "Q39_biggest_barrier",
    "Nếu CI/CD được áp dụng trong các dự án học tập hoặc project cá nhân, bạn mong đợi lợi ích nào nhất? ": "Q40_expected_benefit",
}

# Áp dụng rename (chỉ rename các cột thực sự tồn tại)
existing_mapping = {k: v for k, v in COLUMN_MAPPING.items() if k in df.columns}
df = df.rename(columns=existing_mapping)

# Kiểm tra cột nào chưa được map
unmapped = [c for c in df.columns if c not in COLUMN_MAPPING.values()]
if unmapped:
    print(f"[WARN]  Có {len(unmapped)} cột chưa được map:")
    for c in unmapped:
        print(f"   - '{c}'")
else:
    print(f" Đã rename thành công tất cả {len(existing_mapping)} cột")

print(f"\n Tên cột mới:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 2: PHÂN LOẠI CỘT theo kiểu dữ liệu
# ══════════════════════════════════════════════════════════

# Cột Likert Scale (1-5) - kiểu số
LIKERT_COLS = [c for c in df.columns if c.startswith("Q") and
               int(c.split("_")[0][1:]) >= 15 and int(c.split("_")[0][1:]) <= 32]

# Cột Single-choice (text) - câu trả lời 1 lựa chọn
SINGLE_CHOICE_COLS = [
    "Q1_university", "Q2_year", "Q3_major", "Q4_product_field",
    "Q5_cicd_awareness", "Q6_automation_level",
    "Q11_deploy_frequency", "Q12_lead_time", "Q13_recovery_time", "Q14_failure_rate",
    "Q38_future_plan",
]

# Cột Checkbox (nhiều đáp án, phân cách bởi dấu phẩy)
CHECKBOX_COLS = [
    "Q7_tools_used", "Q8_learning_source", "Q9_usage_purpose", "Q10_cicd_benefits",
    "Q33_cicd_difficulties", "Q34_adoption_barriers",
    "Q39_biggest_barrier", "Q40_expected_benefit",
]

# Cột Free-text
FREE_TEXT_COLS = [
    "Q35_improvement_suggestions", "Q36_university_suggestions", "Q37_skills_needed",
]

print(f" Phân loại cột:")
print(f"  • Likert (1-5):     {len(LIKERT_COLS)} cột")
print(f"  • Single-choice:    {len(SINGLE_CHOICE_COLS)} cột")
print(f"  • Checkbox:         {len(CHECKBOX_COLS)} cột")
print(f"  • Free-text:        {len(FREE_TEXT_COLS)} cột")
print(f"  • Khác (timestamp): 1 cột")

In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 3: XỬ LÝ MISSING VALUES
# ══════════════════════════════════════════════════════════

# 3.1 Thống kê missing trước xử lý
missing_before = df.isnull().sum()
missing_pct = (missing_before / len(df) * 100).round(1)

missing_report = pd.DataFrame({
    "missing_count": missing_before,
    "missing_pct": missing_pct
}).query("missing_count > 0").sort_values("missing_count", ascending=False)

if missing_report.empty:
    print(" Không có missing values!")
else:
    print(f"  Missing values report ({len(missing_report)} cột có missing):")
    print(missing_report.to_string())

In [ ]:
# 3.2 Xử lý missing theo từng loại cột

# LƯU Ý: Không tự động điền giá trị trung vị cho các cột Likert của sinh viên chưa dùng CI/CD
# Điều này để tránh tạo dữ liệu khống làm sai lệch phân tích tương quan và hồi quy UTAUT/DORA chéo.
for col in LIKERT_COLS:
    if col in df.columns:
        n_missing = df[col].isnull().sum()
        print(f"   {col}: giữ nguyên {n_missing} NaN (không điền median)")

# Single-choice & Checkbox: fill NaN với "Không trả lời"
text_cols = SINGLE_CHOICE_COLS + CHECKBOX_COLS + FREE_TEXT_COLS
for col in text_cols:
    if col in df.columns and df[col].isnull().any():
        n_missing = df[col].isnull().sum()
        df[col] = df[col].fillna("Không trả lời")
        print(f"   {col}: filled {n_missing} NaN → 'Không trả lời'")

# Verify
remaining = df.isnull().sum().sum()
print(f"\n Remaining NaN after cleanup: {remaining}")

In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 4: CHUẨN HÓA KIỂU DỮ LIỆU
# ══════════════════════════════════════════════════════════

# Chuyển sang kiểu Int64 (nullable integer) để hỗ trợ NaNs
for col in LIKERT_COLS:
    if col in df.columns:
        df[col] = df[col].astype("Int64")

# Strip whitespace cho tất cả cột text
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

print(" Đã chuẩn hóa kiểu dữ liệu")
print(f"\n Dtypes summary:")
print(df.dtypes.value_counts().to_string())

In [ ]:
# ── Sửa lỗi chính tả tiếng Việt trong dữ liệu khảo sát ──
TYPO_MAP = {
    "Kiểm thử tự đông (Automated Testing)": "Kiểm thử tự động (Automated Testing)",
    "Phân phối liên tuc (CD - Delivery)": "Phân phối liên tục (CD - Delivery)",
    "Xây dụng (Build)": "Xây dựng (Build)",
    "Chưa hiểu rõ lợi ích thực tế khi trển khai CI/CD": "Chưa hiểu rõ lợi ích thực tế khi triển khai CI/CD"
}
for col in df.columns:
    if df[col].dtype == "object":
        for k, v in TYPO_MAP.items():
            df[col] = df[col].str.replace(k, v, regex=False)

# ══════════════════════════════════════════════════════════
# BƯỚC 5: LƯU FILE
# ══════════════════════════════════════════════════════════

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# Verify
verify = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
assert verify.shape == df.shape
print(f" Đã lưu: {OUTPUT_FILE}")
print(f"   Shape: {verify.shape[0]} dòng × {verify.shape[1]} cột")
print(f"   Missing total: {verify.isnull().sum().sum()}")

In [ ]:
# ── Quick sanity check ────────────────────────────────
df.describe(include="all").T.head(15)